# Lab 6: Quantum Fourier Transform (QFT)

**Course:** SCI-104 – Mathematical Foundations and Quantum Mechanics Essentials  
**Student:** Yamalluddin Chahama  
**Student ID:** 6930614038  

This notebook implements and tests the Quantum Fourier Transform using Qiskit.

## Objectives

1. Implement QFT using Hadamard and controlled-phase gates.
2. Test QFT on 2-, 3- and 4-qubit registers.
3. Inspect statevectors and measurement probabilities.
4. Verify that QFT followed by inverse QFT returns the input state.
5. Count the gates used by the direct QFT construction.

In [ ]:
%pip install -q qiskit qiskit-aer pylatexenc

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

SHOTS = 1024
SEED = 42
print('Qiskit environment ready')

## 1. QFT Circuit Function

For an `n`-qubit register, the circuit applies one Hadamard gate to each qubit, controlled-phase rotations with angles `pi / 2**(k-j)`, and final SWAP gates to reverse the qubit order.

In [ ]:
def qft(n):
    """Return the standard n-qubit Quantum Fourier Transform circuit."""
    qc = QuantumCircuit(n, name=f'QFT{n}')

    for j in range(n):
        qc.h(j)
        for k in range(j + 1, n):
            angle = np.pi / (2 ** (k - j))
            qc.cp(angle, k, j)

    # Reverse the qubit order for the conventional QFT output.
    for j in range(n // 2):
        qc.swap(j, n - j - 1)

    return qc

## 2. Display 2-, 3- and 4-Qubit QFT Circuits

In [ ]:
for n in [2, 3, 4]:
    print(f'\n{n}-qubit QFT')
    display(qft(n).draw('mpl'))

In [ ]:
for n in [2, 3, 4]:
    circuit = qft(n)
    print(f'{n}-qubit QFT: depth={circuit.depth()}, gates={dict(circuit.count_ops())}')

## 3. Statevector Test for a 3-Qubit QFT

For a computational-basis input `|j>`, the theoretical output is

$$\mathrm{QFT}_N|j> = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N}|k>.$$

The magnitude of every amplitude should be `1/sqrt(8)` for a 3-qubit input. The input value is represented by relative phases.

In [ ]:
def basis_input(n, value):
    """Create |value> on n qubits using Qiskit's little-endian convention."""
    qc = QuantumCircuit(n)
    for q in range(n):
        if value & (1 << q):
            qc.x(q)
    return qc

n = 3
qft3 = qft(n)

for value in range(2 ** n):
    test = basis_input(n, value)
    test.compose(qft3, inplace=True)
    state = Statevector.from_instruction(test)
    magnitudes = np.abs(state.data)
    print(f'Input |{value:03b}>: amplitudes={np.round(state.data, 4)}')
    print(f'  all magnitudes equal 1/sqrt(8): {np.allclose(magnitudes, 1/np.sqrt(8))}')

In [ ]:
# QFT(|000>) should be a uniform superposition.
zero_input = basis_input(3, 0)
zero_input.compose(qft(3), inplace=True)
zero_state = Statevector.from_instruction(zero_input)

expected = np.ones(8, dtype=complex) / np.sqrt(8)
print('Statevector:', np.round(zero_state.data, 4))
print('Matches theoretical QFT(|000>):', np.allclose(zero_state.data, expected))

## 4. Measurement Test

For a basis-state input, QFT produces equal computational-basis probabilities. With a finite number of shots, the counts are close to but not exactly equal.

In [ ]:
measured = QuantumCircuit(3, 3)
measured.compose(qft(3), inplace=True)
measured.measure(range(3), range(3))

backend = AerSimulator()
result = backend.run(measured, shots=SHOTS, seed_simulator=SEED).result()
counts = result.get_counts()
print(counts)
plot_histogram(counts, title='QFT(|000>) measurement results')

## 5. Inverse QFT Check

QFT is unitary. Therefore, applying QFT followed by its inverse should return the original state.

In [ ]:
initial_circuit = basis_input(3, 5)
initial_state = Statevector.from_instruction(initial_circuit)

round_trip = basis_input(3, 5)
round_trip.compose(qft(3), inplace=True)
round_trip.compose(qft(3).inverse(), inplace=True)
final_state = Statevector.from_instruction(round_trip)

print('Initial state:', initial_state)
print('Final state:  ', final_state)
print('Equivalent:', initial_state.equiv(final_state))

## 6. Effect of Omitting the Final SWAP Gates

The following circuit implements the same QFT decomposition without the final swaps. The transform is still valid, but the qubit order is reversed.

In [ ]:
def qft_without_swaps(n):
    qc = QuantumCircuit(n, name=f'QFT{n}_no_swaps')
    for j in range(n):
        qc.h(j)
        for k in range(j + 1, n):
            qc.cp(np.pi / (2 ** (k - j)), k, j)
    return qc

display(qft_without_swaps(3).draw('mpl'))

## 7. Conclusion

The notebook constructs the QFT directly from elementary gates, verifies the expected statevector amplitudes, confirms the inverse-QFT property, and demonstrates the role of the final SWAP gates. The direct n-qubit QFT uses `n` Hadamard gates, `n(n-1)/2` controlled-phase gates and `floor(n/2)` SWAP gates.